In [ ]:
import mo_gymnasium as mo_gym
import numpy as np
import envs
import csv
from src.modules.commun import Constant
from src.modules.Tools import Tools
#from src.scripts.delete_wandb_dir import delete_directory
from morl_baselines.multi_policy.multi_policy_moqlearning.mp_mo_q_learning import (
    MPMOQLearning,
)

from morl_baselines.multi_policy.pcn.pcn import PCN

GAMMA = 1


# IMPORTANT____________________________________________________________________________________________
# Init Env by giving it the service querry to optimize + the folder where to find the PREPROCESSED data 
serviceIds = [0 , 1 , 2 ]
number_services = len(serviceIds)
MultiCloud_data_dir="./src/data/preprocessedData/NC_20_NS_50/NC_20_NS_50_01"
#______________________________________________________________________________________________________


GAMMA = 1
env = mo_gym.MORecordEpisodeStatistics(mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds ), gamma=GAMMA)
eval_env = mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds )


#### Test env with random actions just to make sure there is no bug in env and the MDP works fine

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = env.action_space.sample()  # this is where you would insert your policy
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

#print("Selected services : ", env.composition.print_composition())

In [ ]:
pf = env.unwrapped.pareto_front()
print(pf)

In [ ]:
timesteps_per_iter: int = 10000

agent = GPIPD(
        env,
        num_nets=2,
        max_grad_norm=None,
        learning_rate=3e-4,
        gamma=0.98,
        batch_size=128,
        net_arch=[256, 256, 256, 256],
        buffer_size=int(2e5),
        initial_epsilon=1.0,
        final_epsilon=0.05,
        epsilon_decay_steps=50000,
        learning_starts=100,
        alpha_per=0.6,
        min_priority=0.01,
        per=gpi_pd,
        gpi_pd=gpi_pd,
        use_gpi=True,
        gradient_updates=g,
        target_net_update_freq=200,
        tau=1,
        dyna=gpi_pd,
        dynamics_uncertainty_threshold=1.5,
        dynamics_net_arch=[256, 256, 256],
        dynamics_buffer_size=int(1e5),
        dynamics_rollout_batch_size=25000,
        dynamics_train_freq=lambda t: 250,
        dynamics_rollout_freq=250,
        dynamics_rollout_starts=5000,
        dynamics_rollout_len=1,
        real_ratio=0.5,
        log=True,
        project_name="GPI_Test",
        experiment_name="GPI-PD",
    )


In [ ]:
agent.train(
        total_timesteps=15 * timesteps_per_iter,
        eval_env=eval_env,
        ref_point=np.array([0.0, 0.0, -200.0]),
        known_pareto_front=env.unwrapped.pareto_front(gamma=0.98),
        weight_selection_algo=algo,
        timesteps_per_iter=timesteps_per_iter,
    )

agent.train(
        eval_env=eval_env,
        total_timesteps=10000,
        ref_point=np.array([-1 ,-1 ,-1 ,-1 ,-1 , -1]),
        num_er_episodes=20,
        max_buffer_size=50,
        num_model_updates=50,
        max_return=np.array(Constant.number_objectives*[number_services]),
        known_pareto_front=pf,

    )

# Use the trained agent with diffrent prefrences  :

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = agent.eval( obs =obs, w = [1/6,1/6,1/6,1/6,1/6,1/6])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)
print("Selected services : ", env.composition.print_composition())


In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/4,1/4,1/4,0,1/4,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())

In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/2,1/4,1/4,0,0,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())